# NB3: Continual learning experiments
**GPU required.** 

This notebook runs three continual learning setups. Later compare them to the baselines from NB2:

- **`ewc_only`:** Train tasks in order with an **EWC** penalty after each task. **No** replay buffer. The model is nudged so weights that mattered on old tasks do not move too freely.
- **`replay_only`:** Train in order with a **replay buffer** only (no EWC). A sample of past examples gets mixed into later training so the optimizer still sees old tasks.
- **`ewc_plus_replay`:** **Both** EWC and replay.

### Inputs (Kaggle: add these as input datasets)
- **`pi-detection-data`** at `/kaggle/input/pi-detection-data/` (parquet files from NB1)
- **`pi-detection-utils`** at `/kaggle/input/pi-detection-utils/` (`utils.py` from your repo)
- **`pi-detection-baselines`** at `/kaggle/input/pi-detection-baselines/` *(optional: attach NB2 output so the big table can include `static_joint` and `naive_sequential`)*

### Outputs (save as a Kaggle dataset, e.g. `pi-detection-cl`)
```
/kaggle/working/
  results/results_cl.json
  checkpoints/ewc_only_after_*.pt
  checkpoints/replay_only_after_*.pt
  checkpoints/ewc_plus_replay_after_*.pt   (NB4 uses these checkpoints)
  replay_buffer/ewc_plus_replay.json         (saved replay buffer)
```

## Install & Setup

In [1]:
!pip install -q transformers accelerate scikit-learn

import sys, os
sys.path.append('/kaggle/input/datasets/dabiraomotoso/pi-detection-utils')
from utils import *

os.makedirs(CFG['checkpoint_dir'], exist_ok=True)
os.makedirs(CFG['results_dir'],    exist_ok=True)
os.makedirs(CFG['replay_dir'],     exist_ok=True)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 1. Load data and build dataloaders
Read the three parquet files from NB1 when they are present. For each task we build **train**, **validation**, and **test** loaders using the split and batch settings in **`utils.CFG`** (same idea as NB2). Missing files are skipped with a message. You generally want all three tasks loaded before you run the experiments below.

In [2]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

In [3]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
data_dir  = '/kaggle/input/datasets/dabiraomotoso/pi-detection-data'
tasks     = {}

for task_name, fname in [
    ('T1_LLMail',      't1_llmail.parquet'),
    ('T2_HackAPrompt', 't2_hackaprompt.parquet'),
    ('T3_BIPIA',       't3_bipia.parquet'),
]:
    path = os.path.join(data_dir, fname)
    if os.path.exists(path):
        df = pd.read_parquet(path)
        tr, va, te = make_loaders(df, tokenizer)
        tasks[task_name] = {'train': tr, 'val': va, 'test': te, 'df': df}
        print(f'  {task_name}: {len(tr.dataset)} train | {len(va.dataset)} val | {len(te.dataset)} test')
    else:
        print(f'  {task_name}: FILE NOT FOUND — skipping')

print(f'Loaded {len(tasks)} tasks.')

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

  T1_LLMail: 14000 train | 3000 val | 3000 test
  T2_HackAPrompt: 14000 train | 3000 val | 3000 test
  T3_BIPIA: 14000 train | 3000 val | 3000 test
Loaded 3 tasks.


In [4]:
import os, gc
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import utils as _utils

def register_task_cpu(self, model, data_loader, task_name):
    print(f'  Computing Fisher matrix for {task_name}...')
    fisher = self.compute_fisher(model, data_loader)
    self.fisher[task_name] = {n: t.cpu() for n, t in fisher.items()}
    self.params[task_name] = {
        n: p.detach().cpu().clone()
        for n, p in model.named_parameters() if p.requires_grad
    }
    self.task_count += 1
    print(f'  EWC registered {task_name} (Fisher on CPU). Total tasks: {self.task_count}')

def penalty_cpu(self, model):
    if self.task_count == 0:
        return torch.tensor(0.0, device=DEVICE)
    loss = torch.tensor(0.0, device=DEVICE)
    for task_name in self.params:
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.fisher[task_name]:
                fisher = self.fisher[task_name][n].to(DEVICE)
                mean   = self.params[task_name][n].to(DEVICE)
                loss  += (fisher * (p - mean) ** 2).sum()
    return self.lam * loss

_utils.EWC.register_task = register_task_cpu
_utils.EWC.penalty       = penalty_cpu

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    print(f'  GPU: {allocated:.2f}GB allocated | {reserved:.2f}GB reserved')

print('Fisher CPU offload active.')
clear_gpu()

Fisher CPU offload active.
  GPU: 0.00GB allocated | 0.00GB reserved


## 2. Experiment 3: EWC only
**Goal:** Train on **T1, then T2, then T3** in order. After each task, **`utils`** registers an **EWC** term: a diagonal Fisher-style importance and a penalty that keeps parameters from drifting too far from the weights that worked on past tasks. **No replay:** you do not store old examples, only this regulariser in weight space.

Watch the printed **per-task test F1** after each stage and the **BWT** line in the summary. The goal is less forgetting than **naive sequential** from NB2. Training time depends on your GPU and **`CFG`**.

In [5]:
all_results = {}

all_results['ewc_only'], _ = run_experiment(
    'ewc_only', tasks, tokenizer,
    use_ewc=True, use_replay=False, joint_training=False,
)
save_results(all_results, f'{CFG["results_dir"]}/results_cl.json')

`torch_dtype` is deprecated! Use `dtype` instead!



EXPERIMENT: ewc_only
  EWC: True | Replay: False | Joint: False


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]


--- Training on T1_LLMail (step 1/3) ---
  Epoch 1/3 | Train Loss: 0.1578 | Val F1: 0.9828 | Val Loss: 0.0788
  Epoch 2/3 | Train Loss: 0.0475 | Val F1: 0.9898 | Val Loss: 0.0456
  Epoch 3/3 | Train Loss: 0.0290 | Val F1: 0.9908 | Val Loss: 0.0473
  Best val F1: 0.9908
  Computing Fisher matrix for T1_LLMail...
  EWC registered T1_LLMail (Fisher on CPU). Total tasks: 1
  After T1_LLMail → T1_LLMail test F1: 0.9943
  Checkpoint saved: /kaggle/working/checkpoints/ewc_only_after_T1_LLMail.pt

--- Training on T2_HackAPrompt (step 2/3) ---
  Epoch 1/3 | Train Loss: 0.7755 | Val F1: 0.5988 | Val Loss: 0.6153
  Epoch 2/3 | Train Loss: 0.5575 | Val F1: 0.7585 | Val Loss: 0.5388
  Epoch 3/3 | Train Loss: 0.4977 | Val F1: 0.7614 | Val Loss: 0.5457
  Best val F1: 0.7614
  Computing Fisher matrix for T2_HackAPrompt...
  EWC registered T2_HackAPrompt (Fisher on CPU). Total tasks: 2
  After T2_HackAPrompt → T1_LLMail test F1: 0.4095
  After T2_HackAPrompt → T2_HackAPrompt test F1: 0.7608
  Checkpoi

In [6]:
clear_gpu()

  GPU: 1.51GB allocated | 1.85GB reserved
